# UAV Battery Tool — Notebook 04: Flight Log Analysis & Parameter Reverse-Engineering

Import real ArduPilot flight logs (.bin/.log/.csv), analyse battery performance, and reverse-engineer electrochemical parameters for improved simulation accuracy.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

from batteries.database import BatteryDatabase
from batteries.log_importer import load_log, generate_synthetic_log
from batteries.parameter_fitter import (
    fit_all, apply_fitted_params,
    fit_internal_resistance, fit_ocv_curve, fit_peukert_k, fit_arrhenius,
)
from mission.simulator import run_simulation

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.25})

DB_PATH = '../battery_db.xlsx'
db = BatteryDatabase(DB_PATH).load()
print(db.summary())

## 1 · Load Flight Log

Change `LOG_PATH` to your actual ArduPilot log. Use `generate_synthetic_log()` for testing.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
USE_SYNTHETIC    = True     # Set False when you have a real log file
LOG_PATH         = '../logs/your_flight.bin'  # .bin, .log, or .csv
PACK_ID          = 'BAT_MID_6S2P'
MISSION_ID       = 'SURVEY_STD'
UAV_ID           = 'HEX_SURVEY_900'
AMBIENT_TEMP_C   = 22.0     # actual temperature during flight
# ─────────────────────────────────────────────────────────────────────────────

pack    = db.packs[PACK_ID]
mission = db.missions[MISSION_ID]
uav     = db.uav_configs[UAV_ID]

if USE_SYNTHETIC:
    print('Using synthetic log (simulated ArduPilot data with noise)...')
    log = generate_synthetic_log(pack, mission, uav, db.discharge_pts,
                                  ambient_temp_c=AMBIENT_TEMP_C, dt_s=2.0,
                                  noise_v=0.035, noise_i=1.2)
else:
    print(f'Loading: {LOG_PATH}')
    log = load_log(LOG_PATH, nominal_capacity_ah=pack.pack_capacity_ah)

print(log.summary())

## 2 · Raw Signal Exploration

In [ ]:
PHASE_COLORS = {'IDLE':'#AAAAAA','TAKEOFF':'#FF9944','CLIMB':'#FFCC44',
    'CRUISE':'#44AA66','HOVER':'#4488FF','DESCEND':'#88AADD','LAND':'#CC88DD',
    'PAYLOAD_OPS':'#FF6688','EMERGENCY':'#FF2222'}
def shade(ax, log_obj):
    if not log_obj.phase_type: return
    prev, t0 = log_obj.phase_type[0], log_obj.time_s[0]
    for t, ph in zip(log_obj.time_s[1:], log_obj.phase_type[1:]):
        if ph != prev:
            ax.axvspan(t0, t, alpha=0.10, color=PHASE_COLORS.get(prev,'#CCC'), zorder=1)
            t0, prev = t, ph
    ax.axvspan(t0, log_obj.time_s[-1], alpha=0.10, color=PHASE_COLORS.get(prev,'#CCC'), zorder=1)

t   = np.array(log.time_s)
v   = np.array(log.voltage_v)
i   = np.array(log.current_a)
mah = np.array(log.mah_used)
tmp = np.array(log.temp_c)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Flight Log — Raw Signals', fontsize=13, fontweight='bold')

axes[0,0].plot(t, v, '#2196F3', linewidth=1.5)
shade(axes[0,0], log)
axes[0,0].set_ylabel('Voltage (V)'); axes[0,0].set_title('Terminal Voltage')

axes[0,1].plot(t, i, '#E53935', linewidth=1.5)
shade(axes[0,1], log)
axes[0,1].set_ylabel('Current (A)'); axes[0,1].set_title('Discharge Current')

axes[1,0].plot(t, mah, '#FF9800', linewidth=1.5)
shade(axes[1,0], log)
axes[1,0].set_ylabel('mAh consumed'); axes[1,0].set_title('Cumulative mAh')

if any(x > -50 for x in log.temp_c):
    axes[1,1].plot(t, tmp, '#9C27B0', linewidth=1.5)
    shade(axes[1,1], log)
    axes[1,1].set_ylabel('Temperature (C)'); axes[1,1].set_title('Cell Temperature')
else:
    axes[1,1].text(0.5, 0.5, 'No temperature data', transform=axes[1,1].transAxes, ha='center')

for ax in axes.flat: ax.set_xlabel('Time (s)')
patches = [mpatches.Patch(color=c, alpha=0.5, label=p) for p, c in PHASE_COLORS.items()
           if p in set(log.phase_type)]
fig.legend(handles=patches, loc='lower center', ncol=6, fontsize=8)
plt.tight_layout(rect=[0,0.05,1,1])
plt.savefig('log_raw_signals.png', bbox_inches='tight'); plt.show()

## 3 · V–I Relationship & IR Fingerprint

Visualise how voltage responds to current — the slope is the internal resistance.

In [ ]:
# Scatter V vs I, coloured by SoC
valid = (np.array(log.current_a) > 2) & (np.array(log.voltage_v) > 5)
i_v = np.array(log.current_a)[valid]
v_v = np.array(log.voltage_v)[valid]
soc_v = np.array(log.soc_pct)[valid] if log.soc_pct else np.linspace(100,0,valid.sum())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('V-I Relationship — Internal Resistance Fingerprint', fontsize=12, fontweight='bold')

sc = axes[0].scatter(i_v, v_v, c=soc_v, cmap='RdYlGn', s=4, alpha=0.5)
plt.colorbar(sc, ax=axes[0], label='SoC (%)')
axes[0].set_xlabel('Current (A)'); axes[0].set_ylabel('Voltage (V)')
axes[0].set_title('V vs I (colour = SoC)')

# IR fit per 30s window
from batteries.parameter_fitter import fit_internal_resistance
r_fit = fit_internal_resistance(log, method='window')
print(f'Fitted R_internal: {r_fit}')

# Overlay best-fit line at median SoC
i_range = np.linspace(i_v.min(), i_v.max(), 50)
v_ocv_est = float(np.median(v_v)) + float(np.median(i_v)) * r_fit.value / 1000
v_fit_line = v_ocv_est - i_range * r_fit.value / 1000
axes[0].plot(i_range, v_fit_line, 'r-', linewidth=2,
             label=f'Fit: R={r_fit.value:.1f}mΩ')
axes[0].legend()

# R² time-series across 30s windows
t_wins = np.arange(log.time_s[0], log.time_s[-1]-30, 15)
r_windows = []
t_arr = np.array(log.time_s)
for tw in t_wins:
    mask = valid & (t_arr >= tw) & (t_arr < tw+30)
    if mask.sum() < 6: r_windows.append(None); continue
    iv, vv = i_v[mask[valid]], v_v[mask[valid]]
    if iv.max()-iv.min() < 0.5: r_windows.append(None); continue
    coeffs = np.polyfit(iv, vv, 1)
    r_windows.append(-coeffs[0]*1000 if coeffs[0] < -0.0005 else None)

r_valid = [(t, r) for t, r in zip(t_wins, r_windows) if r and 0 < r < 400]
if r_valid:
    t_rv, r_rv = zip(*r_valid)
    axes[1].scatter(t_rv, r_rv, s=20, color='steelblue', alpha=0.7)
    axes[1].axhline(r_fit.value, color='red', linewidth=2, label=f'Median: {r_fit.value:.1f}mΩ')
    axes[1].axhline(pack.internal_resistance_mohm, color='green', linewidth=1.5,
                    linestyle='--', label=f'Catalog: {pack.internal_resistance_mohm}mΩ')
    axes[1].set_xlabel('Time (s)'); axes[1].set_ylabel('Fitted R (mΩ)')
    axes[1].set_title('Internal Resistance per 30-s Window')
    axes[1].legend()

plt.tight_layout(); plt.savefig('log_vi_analysis.png', bbox_inches='tight'); plt.show()

## 4 · Full Parameter Fitting Pipeline

In [ ]:
print('Running full parameter fitting pipeline...')
fitted = fit_all(
    log=log,
    nominal_capacity_ah=pack.pack_capacity_ah,
    voltage_cutoff_v_pack=pack.pack_voltage_cutoff,
    chem_id=pack.chemistry_id,
    pack_id=pack.battery_id,
)
print()
print(fitted.summary())

## 5 · OCV Curve Comparison

In [ ]:
from batteries.discharge import DischargeCurve, available_c_rates, closest_c_rate

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('OCV Curve: Fitted from Log vs Catalog', fontsize=12, fontweight='bold')

if fitted.ocv_soc_points:
    ax.plot(fitted.ocv_soc_points, fitted.ocv_voltage_points, 'ro-',
            linewidth=2, markersize=6, label='Fitted from log (cell-level)')

avail = available_c_rates(db.discharge_pts, pack.chemistry_id)
if avail:
    cr = closest_c_rate(avail, 0.5)
    curve = DischargeCurve(db.discharge_pts, pack.chemistry_id, cr, 25.0)
    soc_c = curve.soc_array()
    v_c   = curve.voltage_array()
    ax.plot(soc_c, v_c, 'b-', linewidth=2, alpha=0.7, label=f'Catalog ({cr}C)')

ax.set_xlabel('SoC (%)'); ax.set_ylabel('OCV / Cell Voltage (V)')
ax.set_title('OCV curve comparison'); ax.legend(); ax.grid(alpha=0.3)
ax.invert_xaxis()
plt.tight_layout(); plt.savefig('ocv_comparison.png', bbox_inches='tight'); plt.show()

## 6 · Simulation Comparison: Catalog vs Fitted Parameters

In [ ]:
pack_fitted = apply_fitted_params(fitted, pack)

r_catalog = run_simulation(pack, mission, uav, db.discharge_pts,
                           ambient_temp_c=AMBIENT_TEMP_C, dt_s=1.0)
r_fitted  = run_simulation(pack_fitted, mission, uav, db.discharge_pts,
                           ambient_temp_c=AMBIENT_TEMP_C, dt_s=1.0)

print('Catalog params simulation:')
print(r_catalog.summary())
print()
print('Fitted params simulation:')
print(r_fitted.summary())

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Simulation: Catalog vs Fitted Params', fontsize=12, fontweight='bold')

axes[0].plot(np.array(r_catalog.time_s), np.array(r_catalog.voltage_v),
             'b-', linewidth=2, label='Catalog')
axes[0].plot(np.array(r_fitted.time_s),  np.array(r_fitted.voltage_v),
             'r--', linewidth=2, label='Fitted')
axes[0].plot(np.array(log.time_s), np.array(log.voltage_v),
             'g:', linewidth=1.5, alpha=0.8, label='Real log')
axes[0].set_title('Voltage'); axes[0].set_ylabel('V'); axes[0].legend(fontsize=8)

axes[1].plot(np.array(r_catalog.time_s), np.array(r_catalog.soc_pct), 'b-', linewidth=2)
axes[1].plot(np.array(r_fitted.time_s),  np.array(r_fitted.soc_pct),  'r--', linewidth=2)
if log.soc_pct:
    axes[1].plot(np.array(log.time_s), np.array(log.soc_pct), 'g:', linewidth=1.5, alpha=0.8)
axes[1].set_title('SoC (%)'); axes[1].set_ylabel('SoC (%)')

dv_cat = np.array(r_catalog.dv_ohmic)+np.array(r_catalog.dv_ct)+np.array(r_catalog.dv_conc)
dv_fit = np.array(r_fitted.dv_ohmic)+np.array(r_fitted.dv_ct)+np.array(r_fitted.dv_conc)
axes[2].plot(np.array(r_catalog.time_s), dv_cat, 'b-', linewidth=2, label='Catalog')
axes[2].plot(np.array(r_fitted.time_s),  dv_fit, 'r--', linewidth=2, label='Fitted')
axes[2].set_title('Total Sag (V)'); axes[2].set_ylabel('Sag (V)'); axes[2].legend(fontsize=8)

for ax in axes: ax.set_xlabel('Time (s)'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig('catalog_vs_fitted.png', bbox_inches='tight'); plt.show()

## 7 · Export Fitted Parameters to Database

Save the fitted pack to the Battery Catalog so it can be used in future simulations.

In [ ]:
SAVE_FITTED_PACK = False   # Set True to write to battery_db.xlsx

pack_fitted.battery_id = f'{pack.battery_id}_FITTED'
pack_fitted.notes = (f'Fitted from log: IR={fitted.r_internal_mohm.value:.1f}mΩ  '
                     f'k={fitted.peukert_k.value:.4f}  '
                     f'degrade={fitted.degradation_pct:.1f}%')

print('Fitted pack summary:')
print(f'  Battery ID  : {pack_fitted.battery_id}')
print(f'  IR (mΩ)     : {pack_fitted.internal_resistance_mohm:.2f}  (catalog: {pack.internal_resistance_mohm})')
print(f'  Capacity    : {pack_fitted.pack_capacity_ah:.3f} Ah  (catalog: {pack.pack_capacity_ah})')
print(f'  Energy      : {pack_fitted.pack_energy_wh:.1f} Wh')

if SAVE_FITTED_PACK:
    db.append_custom_pack(pack_fitted)
    print(f'Saved to {DB_PATH}')
else:
    print('Set SAVE_FITTED_PACK = True to persist to battery_db.xlsx')